# Double Pendulum Animation

### Imports

In [27]:
import numpy as np
import matplotlib.pyplot as plt
import celluloid as c

## Pendulum Class

### Equations of Motion

#### Angular Velocity of the Pendulum
$$\dot{\theta} \approx \dot{\theta}_{prev} + \ddot{\theta} \Delta t$$

#### Angular Displacement of the Pendulum
$$\theta \approx \theta_{prev} + \dot{\theta} \Delta t$$

In [28]:
class Pendulum():
    def __init__(self,theta:float,theta_dot:float,mass:float,length:float):
        self.theta = theta
        self.theta_dot = theta_dot
        self.mass = mass
        self.length = length
        self.theta_double_dot = 0

        # Cartesian Coordinates
        self.x = np.sin(self.theta)
        self.y = -np.cos(self.theta)



    def update_velocity(self,dt=0.01):
        """Updates the velocity using basic numerical integration"""
        self.theta_dot += self.theta_double_dot * dt
    
    def update_theta(self,dt=0.01):
        """Updates the value of theta using basic numerical integration"""
        self.theta += self.theta_dot * dt

    def update_position(self,x_1=0,y_1=0):
        self.x = x_1 + self.length * np.sin(self.theta)
        self.y = y_1 - self.length * np.cos(self.theta)


## Angular Acceleration

In [29]:
def get_pendulum_1_angular_acceleration(pendulum1:Pendulum,pendulum2:Pendulum,g=9.81)-> float:

    denominator = (pendulum1.length * 2 * pendulum1.mass) +  pendulum2.mass * np.cos(2*(pendulum1.theta-pendulum2.theta))


    numerator_component1 = -2*g*(pendulum1.mass+pendulum2.mass)*np.sin(pendulum1.theta)
    numerator_component2 = -pendulum2.mass*g*np.sin(pendulum1.theta - pendulum2.theta)
    numerator_component3 = -2*np.sin(pendulum1.theta - pendulum2.theta) * pendulum2.mass * (pendulum1.theta_dot**2) * pendulum2.length
    numerator_component4 = pendulum1.theta_dot**2 * pendulum1.length * np.cos(pendulum1.theta - pendulum2.theta)

    numerator = numerator_component1 + numerator_component2 + numerator_component3 + numerator_component4

    return numerator/denominator

In [30]:
def get_pendulum_2_angular_acceleration(pendulum1:Pendulum,pendulum2:Pendulum,g=9.81)->float:
    combined_mass = pendulum1.mass + pendulum2.mass

    denominator = (pendulum2.length * 2 * pendulum1.mass) - pendulum2.mass * np.cos(2 * (pendulum1.theta - pendulum2.theta))

    numerator_component1 = 2 * np.sin(pendulum1.theta - pendulum2.theta) * (pendulum1.theta_dot**2) * pendulum1.length * combined_mass
    numerator_component2 = g * combined_mass * np.cos(pendulum1.theta) 
    numerator_component3 = (pendulum2.theta_dot**2) * pendulum2.length * pendulum2.mass * np.cos(pendulum1.theta - pendulum2.theta)
    numerator = numerator_component1 + numerator_component2 + numerator_component3

    return numerator/denominator

## Create Two Pendulum Objects To Store Attributes

In [31]:
top_pendulum = Pendulum(0,0,1,1)
bottom_pendulum = Pendulum(0,0,1,1)

## Simulation

In [32]:
duration = 3 # Seconds
dt = 0.01 # Seconds
step_count = int(duration/dt)


top_pendulum_positions = np.zeros((step_count,2))
bottom_pendulum_positions = np.zeros((step_count,2))

In [33]:
# Loop over duration
for time in range(step_count):
    # Calculate Angular Accelerations
    top_pendulum.theta_double_dot = get_pendulum_1_angular_acceleration(top_pendulum,bottom_pendulum)
    bottom_pendulum.theta_double_dot = get_pendulum_2_angular_acceleration(top_pendulum,bottom_pendulum)

    # Update Velocity
    top_pendulum.update_velocity(dt)
    bottom_pendulum.update_velocity(dt)

    # Update Angular Position
    top_pendulum.update_theta(dt)
    bottom_pendulum.update_theta(dt)

    #Store Cartesian Coordinate Information
    top_pendulum.update_position()
    bottom_pendulum.update_position(top_pendulum.x,top_pendulum.y)

    ## Add values to array storing positions

    top_pendulum_positions[time] = np.array([top_pendulum.x,top_pendulum.y])
    bottom_pendulum_positions[time] = np.array([bottom_pendulum.x,bottom_pendulum.y])

    


## Plot Data

In [ ]:
fig = plt.figure(figsize=(6,6))

# To Animate the Figure
cam = c.Camera(fig)

# Create an array to hold previous values so that a trail can be plotted
trail_size = 100
trail = np.ones((trail_size,2))
# Multiply by NaN --> NaN isn't plotted, so any values not overwritten don't get plotted
trail *= np.nan 

for step in range(step_count):
    # Plot Top Pendulum Length
    plt.plot([0,top_pendulum_positions[step,0]],[0,top_pendulum_positions[step,1]],color="blue",linewidth=4)

    # Plot Bottom Pendulum Length
    plt.plot([top_pendulum_positions[step,0],bottom_pendulum_positions[step,0]],[top_pendulum_positions[step,1],bottom_pendulum_positions[step,1]],color="red",linewidth=4)

    # Update Trail
    trail = np.roll(trail,(1,1))

    ## Because always working on the 0 index for trail, update with current positions of lower pendulum
    trail[0,0] = bottom_pendulum_positions[time,0]
    trail[0,1] = bottom_pendulum_positions[time,1]
    
    # Plot Trail
    plt.plot(trail[:,0],trail[:,1],color='grey',zorder=-999)

    cam.snap()


# Animate and Save the Plot
anim = cam.animate(interval=1,repeat=True)
anim.save("Double_Pendulum_Animation.gif")
